# CLIF Validation - Full DQA (Conformance, Completeness, Plausibility, Relational)

In [ ]:
import sys, os
import pandas as pd

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from modules.tables import (
    PatientAnalyzer,
    HospitalizationAnalyzer,
    ADTAnalyzer,
    CodeStatusAnalyzer,
    CRRTTherapyAnalyzer,
    ECMOMCSAnalyzer,
    HospitalDiagnosisAnalyzer,
    LabsAnalyzer,
    MedicationAdminContinuousAnalyzer,
    MedicationAdminIntermittentAnalyzer,
    MicrobiologyCultureAnalyzer,
    MicrobiologyNoncultureAnalyzer,
    MicrobiologySusceptibilityAnalyzer,
    PatientAssessmentsAnalyzer,
    PatientProceduresAnalyzer,
    PositionAnalyzer,
    RespiratorySupportAnalyzer,
    VitalsAnalyzer,
)

In [ ]:
# ---- CONFIG ----
DATA_DIR = '/Users/dema/WD/clifpy/clifpy/data/clif_demo'
FILETYPE = 'parquet'
TIMEZONE = 'UTC'
OUTPUT_DIR = 'dev/output'

In [ ]:
ANALYZERS = [
    ('Patient', PatientAnalyzer),
    ('Hospitalization', HospitalizationAnalyzer),
    ('ADT', ADTAnalyzer),
    ('Code Status', CodeStatusAnalyzer),
    ('CRRT Therapy', CRRTTherapyAnalyzer),
    ('ECMO/MCS', ECMOMCSAnalyzer),
    ('Hospital Diagnosis', HospitalDiagnosisAnalyzer),
    ('Labs', LabsAnalyzer),
    ('Medication Admin Continuous', MedicationAdminContinuousAnalyzer),
    ('Medication Admin Intermittent', MedicationAdminIntermittentAnalyzer),
    ('Microbiology Culture', MicrobiologyCultureAnalyzer),
    ('Microbiology Nonculture', MicrobiologyNoncultureAnalyzer),
    ('Microbiology Susceptibility', MicrobiologySusceptibilityAnalyzer),
    ('Patient Assessments', PatientAssessmentsAnalyzer),
    ('Patient Procedures', PatientProceduresAnalyzer),
    ('Position', PositionAnalyzer),
    ('Respiratory Support', RespiratorySupportAnalyzer),
    ('Vitals', VitalsAnalyzer),
]

DQA_CATEGORIES = ('conformance', 'completeness', 'relational', 'plausibility')

def _format_error(e):
    """Format an error message, including details like invalid values or missingness."""
    msg = e['message']
    details = e.get('details', {})
    if 'top_invalid' in details:
        vals = ', '.join(f"{iv['value']} (n={iv['count']})" for iv in details['top_invalid'])
        msg += f" — invalid: {vals}"
    if 'percent_missing' in details:
        msg += f" — {details['percent_missing']:.1f}% missing"
    return msg


def build_detail_df(dqa_result):
    """Build a DataFrame with one row per validation check from run_full_dqa output."""
    rows = []
    for category in DQA_CATEGORIES:
        checks = dqa_result.get(category, {})
        if not checks:
            continue
        for check_name, d in checks.items():
            error_msgs = '; '.join(_format_error(e) for e in d['errors']) if d['errors'] else ''
            warning_msgs = '; '.join(_format_error(w) for w in d['warnings']) if d['warnings'] else ''
            metrics_str = ', '.join(f"{k}={v}" for k, v in d['metrics'].items()
                                    if not isinstance(v, (list, dict))) if d['metrics'] else ''
            rows.append({
                'Check': d['check_type'],
                'Category': category.title(),
                'Passed': d['passed'],
                'Errors': len(d['errors']),
                'Warnings': len(d['warnings']),
                'Details': error_msgs or warning_msgs or '',
                'Metrics': metrics_str,
            })
    return pd.DataFrame(rows)

## Per-Table Validation Details

In [ ]:
# --- Pass 1: Load all analyzers and collect tables ---
all_results = {}
loaded_tables = []

for name, AnalyzerClass in ANALYZERS:
    try:
        analyzer = AnalyzerClass(
            data_dir=DATA_DIR,
            filetype=FILETYPE,
            timezone=TIMEZONE,
            output_dir=OUTPUT_DIR,
        )
        if analyzer.table is None:
            print(f"  {name}: No data file found, skipping.")
            all_results[name] = {'status': 'skipped'}
            continue

        all_results[name] = {
            'status': 'loaded',
            'analyzer': analyzer,
            'shape': analyzer.table.df.shape,
        }
        loaded_tables.append(analyzer.table)

    except Exception as e:
        print(f"  {name} ERROR: {e}")
        all_results[name] = {'status': 'error', 'error': str(e)}

print(f"Loaded {len(loaded_tables)} / {len(ANALYZERS)} tables")

In [ ]:
# --- Pass 2: Run full DQA (conformance + completeness + relational + plausibility) ---
for name, info in all_results.items():
    if info['status'] != 'loaded':
        continue

    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    analyzer = info['analyzer']
    print(f"  Shape: {info['shape']}")

    try:
        dqa_result = analyzer.validate(tables=loaded_tables)

        detail_df = build_detail_df(dqa_result)
        detail_df.to_csv(f"output/{name}_dqa.csv", index=False)
        display(detail_df)

        info['status'] = 'ok'
        info['dqa_result'] = dqa_result

    except Exception as e:
        print(f"  ERROR during DQA: {e}")
        info['status'] = 'error'
        info['error'] = str(e)

## Overall Summary

In [ ]:
def _score(checks):
    """Summarize pass/total for a dict of check results."""
    if not checks:
        return 'N/A'
    total = len(checks)
    passed = sum(1 for v in checks.values() if v['passed'])
    return f"{passed}/{total}"

summary_rows = []
for name, info in all_results.items():
    if info['status'] in ('skipped', 'error'):
        summary_rows.append({
            'Table': name,
            'Status': info['status'].upper(),
            'Rows': None,
            'Cols': None,
            'Conformance': None,
            'Completeness': None,
            'Plausibility': None,
            'Relational': None,
        })
    else:
        r, c = info['shape']
        dqa = info['dqa_result']
        summary_rows.append({
            'Table': name,
            'Status': 'OK',
            'Rows': r,
            'Cols': c,
            'Conformance': _score(dqa.get('conformance', {})),
            'Completeness': _score(dqa.get('completeness', {})),
            'Plausibility': _score(dqa.get('plausibility', {})),
            'Relational': _score(dqa.get('relational', {})),
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)